In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv("../.env")
TOKEN = os.getenv("TINKOFF_TOKEN")

BASE_URL = "https://invest-public-api.tinkoff.ru/rest"
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

print(f"Токен загружен: {'✅' if TOKEN else '❌'}")

Токен загружен: ✅


In [3]:
TICKERS = ["SBER", "GAZP", "LKOH", "GMKN", "NVTK",
           "TATN", "MGNT", "ROSN", "MTSS", "ALRS",
           "CHMF", "NLMK", "PIKK", "VTBR", "PHOR"]

uid_map = {}

for ticker in TICKERS:
    resp = requests.post(
        f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/FindInstrument",
        headers=HEADERS,
        json={"query": ticker, "instrumentKind": "INSTRUMENT_TYPE_SHARE", "apiTradeAvailableFlag": True}
    )
    data = resp.json()
    for inst in data.get("instruments", []):
        if inst.get("ticker") == ticker:
            uid_map[ticker] = inst["uid"]
            print(f"{ticker} → {inst['uid']}")
            break

print(f"\nНайдено: {len(uid_map)}/{len(TICKERS)}")

SBER → e6123145-9665-43e0-8413-cd61b8aa9b13
GAZP → 962e2a95-02a9-4171-abd7-aa198dbe643a
LKOH → 02cfdf61-6298-4c0f-a9ca-9cabc82afaf3
GMKN → 509edd0c-129c-4ee2-934d-7f6246126da1
NVTK → 0da66728-6c30-44c4-9264-df8fac2467ee
TATN → 88468f6c-c67a-4fb4-a006-53eed803883c
MGNT → ca845f68-6c43-44bc-b584-330d2a1e5eb7
ROSN → fd417230-19cf-4e7b-9623-f7c9ca18ec6b
MTSS → cd8063ad-73ad-4b31-bd0d-93138d9e99a2
ALRS → 30817fea-20e6-4fee-ab1f-d20fc1a1bb72
CHMF → fa6aae10-b8d5-48c8-bbfd-d320d925d096
NLMK → 161eb0d0-aaac-4451-b374-f5d0eeb1b508
PIKK → 03d5e771-fc10-438e-8892-85a40733612d
VTBR → 8e2b0325-0292-4654-8a18-4f63ed3b0e09
PHOR → 9978b56f-782a-4a80-a4b1-a48cbecfd194

Найдено: 15/15


In [4]:
ticker_to_asset_uid = {}

for ticker, inst_uid in uid_map.items():
    resp = requests.post(
        f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetInstrumentBy",
        headers=HEADERS,
        json={"idType": "INSTRUMENT_ID_TYPE_UID", "id": inst_uid}
    )
    data = resp.json()
    asset_uid = data.get("instrument", {}).get("assetUid", "")
    ticker_to_asset_uid[ticker] = asset_uid
    print(f"{ticker} → asset_uid: {asset_uid}")

print(f"\nНайдено: {len(ticker_to_asset_uid)}")

SBER → asset_uid: 40d89385-a03a-4659-bf4e-d3ecba011782
GAZP → asset_uid: bfc8184d-9562-4ea2-87dd-be6e76dc1279
LKOH → asset_uid: f898047a-dac8-4717-a40f-7f11211139ef
GMKN → asset_uid: ba7b5a1b-8515-4134-8c26-b69a2372fe82
NVTK → asset_uid: 2d850de1-466d-443d-b7fd-e78bf41146fb
TATN → asset_uid: 9da0b54d-f8dd-4e75-9081-ec0b8257775c
MGNT → asset_uid: 4833e124-265f-4879-adc8-58bdf983f54e
ROSN → asset_uid: 47ba7e01-8f52-4723-90e5-3ea379b9ccae
MTSS → asset_uid: eb85ff63-713d-4ace-b4c8-922c9a5fc812
ALRS → asset_uid: f924d7df-b6d0-4e36-9bf7-f38aaac2ce53
CHMF → asset_uid: e72889a3-21db-4b2e-90bc-045ec3d28d04
NLMK → asset_uid: fb0fb8a8-fcd5-4f78-b5aa-73b3be9f4454
PIKK → asset_uid: c14ab000-c378-432d-9686-1055170a4c3d
VTBR → asset_uid: 94f18cc6-9e01-4db6-9d73-8c3ba6ab9655
PHOR → asset_uid: 5a3d1efd-f8a0-478e-a10e-bb7f990f9c87

Найдено: 15


In [5]:
asset_uids = list(ticker_to_asset_uid.values())

resp = requests.post(
    f"{BASE_URL}/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetAssetFundamentals",
    headers=HEADERS,
    json={"assets": asset_uids}
)
data = resp.json()

records = []
asset_uid_to_ticker = {v: k for k, v in ticker_to_asset_uid.items()}

for f in data.get("fundamentals", []):
    ticker = asset_uid_to_ticker.get(f.get("assetUid", ""), "UNKNOWN")
    records.append({
        "ticker":            ticker,
        "pe_ratio":          f.get("peRatioTtm"),
        "pb_ratio":          f.get("priceToBookTtm"),
        "ps_ratio":          f.get("priceToSalesTtm"),
        "ev_ebitda":         f.get("evToEbitdaMrq"),
        "roe":               f.get("roe"),
        "roa":               f.get("roa"),
        "net_margin":        f.get("netMarginMrq"),
        "debt_to_equity":    f.get("totalDebtToEquityMrq"),
        "debt_to_ebitda":    f.get("totalDebtToEbitdaMrq"),
        "net_debt_ebitda":   f.get("netDebtToEbitda"),
        "current_ratio":     f.get("currentRatioMrq"),
        "div_yield":         f.get("dividendYieldDailyTtm"),
        "revenue_growth_1y": f.get("oneYearAnnualRevenueGrowthRate"),
        "revenue_growth_3y": f.get("threeYearAnnualRevenueGrowthRate"),
        "revenue_growth_5y": f.get("fiveYearAnnualRevenueGrowthRate"),
        "eps_ttm":           f.get("epsTtm"),
        "beta":              f.get("beta"),
        "market_cap":        f.get("marketCapitalization"),
    })

df_fund = pd.DataFrame(records)

In [6]:
print(df_fund.to_string())
print("\nПропуски:")
print(df_fund.isnull().sum())

   ticker  pe_ratio  pb_ratio  ps_ratio  ev_ebitda     roe    roa  net_margin  debt_to_equity  debt_to_ebitda  net_debt_ebitda  current_ratio  div_yield  revenue_growth_1y  revenue_growth_3y  revenue_growth_5y  eps_ttm  beta    market_cap
0    SBER      4.01      0.82      0.63       0.00   22.19   2.70        0.00            0.00             0.0             0.00            0.0      11.01              25.20              33.00                0.0    79.02  0.57  6.833348e+12
1    GAZP      2.10      0.16      0.29       0.00    7.95   4.71       13.91           35.17             0.0             0.00            0.0       0.00              25.44               1.52                0.0    60.98  1.03  3.036128e+12
2    LKOH      0.00      1.04      1.00       3.96  -19.71 -14.13      -28.13            8.77             0.0            -0.25            0.0      17.27             -56.30               0.00                0.0 -1529.67  0.88  3.762261e+12
3    GMKN      9.76      1.92      1.75     

In [7]:
import numpy as np

def score_metric(value, good, bad, higher_is_better=True):
    """Нормализует метрику в диапазон 0-100"""
    if value is None or np.isnan(value):
        return 50  # нейтральный балл если нет данных
    if higher_is_better:
        if value >= good: return 100
        if value <= bad:  return 0
        return (value - bad) / (good - bad) * 100
    else:
        if value <= good: return 100
        if value >= bad:  return 0
        return (bad - value) / (bad - good) * 100

def fundamental_score(row):
    scores = {}

    # P/E — чем ниже тем лучше, 0 = нет данных
    if row["pe_ratio"] > 0:
        scores["pe"] = score_metric(row["pe_ratio"], good=5, bad=25, higher_is_better=False)
    else:
        scores["pe"] = 50

    # P/B — чем ниже тем лучше
    if row["pb_ratio"] > 0:
        scores["pb"] = score_metric(row["pb_ratio"], good=0.5, bad=4, higher_is_better=False)
    else:
        scores["pb"] = 50

    # ROE — чем выше тем лучше, аномалии отсекаем
    roe = row["roe"]
    if roe > 100 or roe < -100:
        scores["roe"] = 50
    else:
        scores["roe"] = score_metric(roe, good=20, bad=0, higher_is_better=True)

    # ROA — чем выше тем лучше
    scores["roa"] = score_metric(row["roa"], good=10, bad=0, higher_is_better=True)

    # Net margin — чем выше тем лучше
    if row["net_margin"] != 0:
        scores["net_margin"] = score_metric(row["net_margin"], good=20, bad=0, higher_is_better=True)
    else:
        scores["net_margin"] = 50

    # D/E — чем ниже тем лучше, аномалии отсекаем
    de = row["debt_to_equity"]
    if de > 500 or de < 0:
        scores["debt"] = 20  # штраф за аномальный долг
    else:
        scores["debt"] = score_metric(de, good=0.5, bad=5, higher_is_better=False)

    # Дивидендная доходность — чем выше тем лучше
    scores["div"] = score_metric(row["div_yield"], good=15, bad=0, higher_is_better=True)

    # Рост выручки 1 год
    scores["growth"] = score_metric(row["revenue_growth_1y"], good=20, bad=-10, higher_is_better=True)

    # EV/EBITDA — чем ниже тем лучше, 0 = нет данных
    if row["ev_ebitda"] > 0:
        scores["ev_ebitda"] = score_metric(row["ev_ebitda"], good=3, bad=12, higher_is_better=False)
    else:
        scores["ev_ebitda"] = 50

    # Веса
    weights = {
        "pe":         0.15,
        "pb":         0.10,
        "roe":        0.15,
        "roa":        0.10,
        "net_margin": 0.10,
        "debt":       0.15,
        "div":        0.10,
        "growth":     0.10,
        "ev_ebitda":  0.05,
    }

    total = sum(scores[k] * weights[k] for k in weights)
    details = {k: round(v, 1) for k, v in scores.items()}
    return round(total, 1), details

# Применяем
results = []
for _, row in df_fund.iterrows():
    score, details = fundamental_score(row)
    results.append({
        "ticker": row["ticker"],
        "score":  score,
        **details
    })

df_scores = pd.DataFrame(results).sort_values("score", ascending=False).reset_index(drop=True)
print(df_scores[["ticker", "score", "pe", "pb", "roe", "roa", "debt", "div", "growth"]].to_string())

   ticker  score     pe     pb    roe    roa   debt    div  growth
0    VTBR   82.6  100.0  100.0   91.6   13.9  100.0  100.0   100.0
1    SBER   81.6  100.0   90.9  100.0   27.0  100.0   73.4   100.0
2    PHOR   63.2   86.3    0.0  100.0  100.0    0.0   34.8    84.3
3    TATN   58.2   81.0   85.4   58.1   71.0   52.4   71.2     0.0
4    GMKN   56.5   76.2   59.4  100.0   85.7    0.0    0.0    30.8
5    GAZP   55.1  100.0  100.0   39.8   47.1    0.0    0.0   100.0
6    ROSN   53.7   80.5   99.1   29.8   26.0  100.0   38.3     0.0
7    ALRS   49.2   92.6   97.7   45.8   53.5    0.0    0.0    27.8
8    MGNT   46.5   71.5   69.4   78.4   17.5    0.0    0.0    98.7
9    MTSS   46.2   68.1    0.0   50.0   23.3   20.0  100.0    82.3
10   NVTK   40.6   26.5   77.4   32.2   50.1    0.0   71.5    11.7
11   NLMK   39.1   80.5   96.3   36.3   56.1    0.0    0.0     0.0
12   LKOH   30.4   50.0   84.6    0.0    0.0    0.0  100.0     0.0
13   PIKK   30.0   37.1   90.0   25.1   12.5    0.0    0.0    

In [8]:
def interpret(score):
    if score >= 70: return "Привлекательна"
    if score >= 50: return "Нейтральна"
    return "Непривлекательна"

df_scores["interpretation"] = df_scores["score"].apply(interpret)
print("\nФундаментальный скоринг:")
print(df_scores[["ticker", "score", "interpretation"]].to_string())


Фундаментальный скоринг:
   ticker  score    interpretation
0    VTBR   82.6    Привлекательна
1    SBER   81.6    Привлекательна
2    PHOR   63.2        Нейтральна
3    TATN   58.2        Нейтральна
4    GMKN   56.5        Нейтральна
5    GAZP   55.1        Нейтральна
6    ROSN   53.7        Нейтральна
7    ALRS   49.2  Непривлекательна
8    MGNT   46.5  Непривлекательна
9    MTSS   46.2  Непривлекательна
10   NVTK   40.6  Непривлекательна
11   NLMK   39.1  Непривлекательна
12   LKOH   30.4  Непривлекательна
13   PIKK   30.0  Непривлекательна
14   CHMF   24.7  Непривлекательна


In [11]:
from scipy.stats import percentileofscore

def percentile_score(series, value, higher_is_better=True):
    """Возвращает перцентиль значения среди всех компаний"""
    # Убираем нули и аномалии
    clean = series.replace(0, np.nan).dropna()
    clean = clean[clean.abs() < 500]  # убираем аномалии типа MTSS D/E=4204
    
    if len(clean) < 2 or np.isnan(value) or value == 0:
        return 50  # нейтральный если нет данных
    
    pct = percentileofscore(clean, value, kind="rank")
    return pct if higher_is_better else 100 - pct

# Метрики и их направление
metrics = {
    "pe_ratio":          (False, 0.15),  # ниже = лучше
    "pb_ratio":          (False, 0.10),  # ниже = лучше
    "roe":               (True,  0.15),  # выше = лучше
    "roa":               (True,  0.10),  # выше = лучше
    "net_margin":        (True,  0.10),  # выше = лучше
    "debt_to_equity":    (False, 0.15),  # ниже = лучше
    "div_yield":         (True,  0.10),  # выше = лучше
    "revenue_growth_1y": (True,  0.10),  # выше = лучше
    "ev_ebitda":         (False, 0.05),  # ниже = лучше
}

records = []
for _, row in df_fund.iterrows():
    scores = {}
    for col, (higher_is_better, weight) in metrics.items():
        scores[col] = percentile_score(df_fund[col], row[col], higher_is_better)
    
    total = sum(scores[col] * weight for col, (_, weight) in metrics.items())
    
    records.append({
        "ticker":   row["ticker"],
        "score":    round(total, 1),
        "pe":       round(scores["pe_ratio"], 1),
        "pb":       round(scores["pb_ratio"], 1),
        "roe":      round(scores["roe"], 1),
        "roa":      round(scores["roa"], 1),
        "margin":   round(scores["net_margin"], 1),
        "debt":     round(scores["debt_to_equity"], 1),
        "div":      round(scores["div_yield"], 1),
        "growth":   round(scores["revenue_growth_1y"], 1),
        "ev_ebitda":round(scores["ev_ebitda"], 1),
    })

df_scores = pd.DataFrame(records).sort_values("score", ascending=False).reset_index(drop=True)

def interpret(score):
    if score >= 65: return "Привлекательна"
    if score >= 40: return "Нейтральна"
    return "Непривлекательна"

df_scores["interpretation"] = df_scores["score"].apply(interpret)

print(df_scores[["ticker", "score", "pe", "pb", "roe", "debt", "div", "growth", "interpretation"]].to_string())

df_scores.to_csv("../data/fundamental_scores.csv", index=False)
print("\nСохранено!")

   ticker  score    pe    pb    roe  debt    div  growth      interpretation
0    VTBR   70.3  92.9  86.7   80.0  50.0  100.0    86.7    🟢 Привлекательна
1    GAZP   68.2  85.7  93.3   53.3  45.5   50.0   100.0    🟢 Привлекательна
2    SBER   67.0  78.6  60.0   93.3  50.0   62.5    93.3    🟢 Привлекательна
3    TATN   61.3  57.1  46.7   66.7  90.9   37.5    33.3        🟡 Нейтральна
4    ALRS   58.6  71.4  73.3   60.0  36.4   50.0    46.7        🟡 Нейтральна
5    PHOR   57.2  64.3   6.7  100.0  18.2   12.5    66.7        🟡 Нейтральна
6    NLMK   55.8  46.4  66.7   46.7  81.8   50.0    20.0        🟡 Нейтральна
7    GMKN   54.6  35.7  13.3   86.7  27.3   50.0    53.3        🟡 Нейтральна
8    NVTK   45.7   7.1  33.3   40.0  63.6   50.0    40.0        🟡 Нейтральна
9    ROSN   41.4  46.4  80.0   26.7  50.0   25.0    13.3        🟡 Нейтральна
10   LKOH   39.0  50.0  40.0   13.3  72.7   87.5     6.7  🔴 Непривлекательна
11   MGNT   38.9  28.6  20.0   73.3  -0.0   50.0    80.0  🔴 Непривлекательна

In [14]:
import numpy as np

def graham_score(row):
    scores = {}

    scores["pe"]         = 1 if 0 < row["pe_ratio"] < 15 else 0
    scores["pb"]         = 1 if 0 < row["pb_ratio"] < 1.5 else 0
    scores["pe_pb"]      = 1 if row["pe_ratio"] > 0 and row["pb_ratio"] > 0 and row["pe_ratio"] * row["pb_ratio"] < 22.5 else 0
    scores["debt"]       = 1 if 0 <= row["debt_to_equity"] < 100 else 0
    scores["liquidity"]  = 1 if row["net_margin"] > 5 else 0
    scores["profitable"] = 1 if row["roa"] > 0 else 0
    scores["dividend"]   = 1 if row["div_yield"] > 0 else 0
    scores["growth"]     = 1 if row["revenue_growth_1y"] > 0 else 0
    scores["roe"]        = 1 if row["roe"] > 10 else 0
    scores["eps"]        = 1 if row["eps_ttm"] > 0 else 0

    total_points = sum(scores.values())
    total_score  = round(total_points / 10 * 100, 1)

    return total_score, total_points, scores

records = []
for _, row in df_fund.iterrows():
    score, points, details = graham_score(row)
    records.append({
        "ticker":     row["ticker"],
        "score":      score,
        "points":     f"{points}/10",
        "pe":         details["pe"],
        "pb":         details["pb"],
        "pe_pb":      details["pe_pb"],
        "debt":       details["debt"],
        "liquidity":  details["liquidity"],
        "profitable": details["profitable"],
        "dividend":   details["dividend"],
        "growth":     details["growth"],
        "roe":        details["roe"],
        "eps":        details["eps"],
    })

df_scores = pd.DataFrame(records).sort_values("score", ascending=False).reset_index(drop=True)

def interpret(score):
    if score >= 70: return "Привлекательна"
    if score >= 50: return "Нейтральна"
    return "Непривлекательна"

df_scores["interpretation"] = df_scores["score"].apply(interpret)

print(df_scores[["ticker", "score", "points", "pe", "pb", "debt",
                  "profitable", "dividend", "growth", "roe", "interpretation"]].to_string())

df_scores.to_csv("../data/fundamental_scores.csv", index=False)

   ticker  score points  pe  pb  debt  profitable  dividend  growth  roe    interpretation
0    SBER   90.0   9/10   1   1     1           1         1       1    1    Привлекательна
1    TATN   90.0   9/10   1   1     1           1         1       0    1    Привлекательна
2    VTBR   90.0   9/10   1   1     1           1         1       1    1    Привлекательна
3    GAZP   80.0   8/10   1   1     1           1         0       1    0    Привлекательна
4    ROSN   80.0   8/10   1   1     1           1         1       0    0    Привлекательна
5    GMKN   70.0   7/10   1   0     1           1         0       0    1    Привлекательна
6    ALRS   70.0   7/10   1   1     1           1         0       0    0    Привлекательна
7    NLMK   70.0   7/10   1   1     1           1         0       0    0    Привлекательна
8    PHOR   70.0   7/10   1   0     0           1         1       1    1    Привлекательна
9    NVTK   60.0   6/10   0   1     1           1         1       0    0        Нейтральна

In [15]:
df_scores.to_csv("../data/fundamental_scores.csv", index=False)
print("Сохранено → ml/data/fundamental_scores.csv")

Сохранено → ml/data/fundamental_scores.csv
